# 01 — Extração de Dados Brutos

**Objetivo:** baixar e armazenar todos os dados brutos necessários para o projeto, sem aplicar transformações ou filtros. As limpezas, classificações e merges acontecem no notebook `02_transformacao.ipynb`.

## Fontes coletadas

| # | Fonte | Acesso | Tabela / Endpoint | Conteúdo |
|---|-------|--------|-------------------|----------|
| 1 | **SIM-DATASUS** | FTP DATASUS | `/dissemin/publicos/SIM/CID10/DORES/` | Microdados de óbitos por UF/ano (DO\*.dbc) |
| 2 | **População Municipal** | IBGE Sidra | Tab 4714 var 93 (Censo 2022) / Tab 6579 var 9324 (estimativas) | População residente por município/ano |
| 3 | **PIB Municipal** | IBGE Sidra | Tab 5938 var 37 | PIB total a preços correntes (R$ mil) |
| 4 | **Despesa em Saúde** | SICONFI / Tesouro Nacional | API DCA-Anexo I-E, função 10 (Saúde) | Despesas empenhadas/liquidadas em saúde por município |
| 5 | **IDH-M, Gini, RDPC** | Atlas Brasil (PNUD/IPEA/FJP) | Download manual | Indicadores socioeconômicos do Censo 2010 |

## Saída

```
data/raw/
├── sim/                              # 1 parquet por UF/ano (DBC original convertido)
│   ├── DOAC2022.parquet
│   └── ...
├── ibge/
│   ├── populacao_municipal_{ano}.csv
│   └── pib_municipal_{ano}.csv
├── siconfi/
│   └── despesa_saude_{ano}.csv       # consolidado de todos os municípios
└── atlas/
    └── atlas_brasil.csv              # download manual
```

In [ ]:
import ftplib
import json
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import requests
from datasus_dbc import decompress as dbc_decompress
from dbfread import DBF

ROOT = Path.cwd().parent
RAW_DIR      = ROOT / "data" / "raw"
SIM_DIR      = RAW_DIR / "sim"
IBGE_DIR     = RAW_DIR / "ibge"
SICONFI_DIR  = RAW_DIR / "siconfi"
ATLAS_DIR    = RAW_DIR / "atlas"
for d in (SIM_DIR, IBGE_DIR, SICONFI_DIR, ATLAS_DIR):
    d.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
print("Imports OK")

: 

## 1. Configuração de escopo

Defina o conjunto de UFs e anos a extrair. **Todas as etapas são idempotentes** — re-executar só baixa o que falta.

| Parâmetro | Valor recomendado | Observação |
|-----------|-------------------|------------|
| `ANOS`    | `[2022]`          | 2022 = ano do Censo, melhor join com IBGE |
| `UFS`     | `UFS_TODAS`       | Todas as 27 UFs para análise nacional |

In [ ]:
UFS_TODAS = [
    "AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO",
    "MA", "MG", "MS", "MT", "PA", "PB", "PE", "PI", "PR",
    "RJ", "RN", "RO", "RR", "RS", "SC", "SE", "SP", "TO",
]

ANOS = [2022]
UFS  = UFS_TODAS

print(f"ANOS : {ANOS}")
print(f"UFS  : {len(UFS)} estados")

## 2. SIM-DATASUS — Microdados de óbitos

Download dos arquivos `DO{UF}{ANO}.dbc` direto do FTP DATASUS, conversão para Parquet sem nenhuma transformação. Cada arquivo é salvo em `data/raw/sim/`.

In [ ]:
FTP_HOST = "ftp.datasus.gov.br"
FTP_DIR  = "/dissemin/publicos/SIM/CID10/DORES"


def baixar_dbcs_sim(ufs, anos, destino: Path) -> list[Path]:
    """Baixa os DOXXAAAA.dbc do FTP reaproveitando uma única conexão."""
    pendentes, cacheados = [], []
    for uf in ufs:
        for ano in anos:
            nome = f"DO{uf.upper()}{ano}.dbc"
            local = destino / nome
            (cacheados if local.exists() else pendentes).append((nome, local))
    print(f"Em cache: {len(cacheados)}  |  A baixar: {len(pendentes)}")
    if pendentes:
        with ftplib.FTP(FTP_HOST, timeout=180) as ftp:
            ftp.login()
            ftp.cwd(FTP_DIR)
            for i, (nome, local) in enumerate(pendentes, 1):
                print(f"  [{i:>2}/{len(pendentes)}] {nome} ...", end=" ", flush=True)
                try:
                    with open(local, "wb") as f:
                        ftp.retrbinary(f"RETR {nome}", f.write)
                    print(f"{local.stat().st_size / 1024:.0f} KB")
                except ftplib.error_perm as e:
                    local.unlink(missing_ok=True)
                    print(f"FALHOU: {e}")
    return [destino / f"DO{uf}{ano}.dbc"
            for uf in ufs for ano in anos
            if (destino / f"DO{uf}{ano}.dbc").exists()]


dbc_files = baixar_dbcs_sim(UFS, ANOS, SIM_DIR)
print(f"\nTotal de DBCs disponíveis: {len(dbc_files)}")

In [ ]:
def dbc_para_parquet(dbc_path: Path) -> Path:
    """
    Converte DBC → DBF (temporário) → DataFrame → Parquet (raw, sem transformações).
    Idempotente: pula se o parquet já existe.
    """
    parquet_path = dbc_path.with_suffix(".parquet")
    if parquet_path.exists():
        return parquet_path

    dbf_path = dbc_path.with_suffix(".dbf")
    dbc_decompress(str(dbc_path), str(dbf_path))
    try:
        table = DBF(str(dbf_path), encoding="latin1", char_decode_errors="replace")
        df = pd.DataFrame(iter(table))
    finally:
        dbf_path.unlink(missing_ok=True)

    df.to_parquet(parquet_path, index=False)
    return parquet_path


for i, dbc in enumerate(dbc_files, 1):
    pq = dbc_para_parquet(dbc)
    n = len(pd.read_parquet(pq, columns=[pd.read_parquet(pq).columns[0]]))
    print(f"  [{i:>2}/{len(dbc_files)}] {pq.name}  → {n:,} registros")

print(f"\n SIM em: {SIM_DIR}")

## 3. População Municipal — IBGE Sidra

- **Ano 2022** → Tabela 4714 / Variável 93 (Censo Demográfico 2022).
- **Outros anos disponíveis (2017–2021, 2024–2025)** → Tabela 6579 / Variável 9324 (Estimativas).

Cada chamada é cacheada em `data/raw/ibge/populacao_municipal_{ano}.csv`.

In [ ]:
def fetch_populacao_municipal(ano: int) -> pd.DataFrame:
    cache = IBGE_DIR / f"populacao_municipal_{ano}.csv"
    if cache.exists():
        return pd.read_csv(cache, dtype={"CODMUN7": str, "CODMUN6": str})

    if ano == 2022:
        url = f"https://apisidra.ibge.gov.br/values/t/4714/n6/all/v/93/p/{ano}"
    else:
        url = f"https://apisidra.ibge.gov.br/values/t/6579/n6/all/v/9324/p/{ano}"

    print(f"  → Sidra: {url}")
    r = requests.get(url, timeout=300)
    r.raise_for_status()
    raw = r.json()
    if not raw or len(raw) < 2:
        raise RuntimeError(f"Resposta vazia do Sidra para {ano}")

    df = pd.DataFrame(raw[1:])
    df = df.rename(columns={"D1C": "CODMUN7", "D1N": "MUNICIPIO_NOME", "V": "POPULACAO"})
    df["POPULACAO"]      = pd.to_numeric(df["POPULACAO"], errors="coerce")
    df["ANO"]            = ano
    df["CODMUN7"]        = df["CODMUN7"].astype(str).str.strip()
    df["CODMUN6"]        = df["CODMUN7"].str[:6]
    df = df[["CODMUN7", "CODMUN6", "MUNICIPIO_NOME", "ANO", "POPULACAO"]]
    df.to_csv(cache, index=False)
    print(f"     {len(df):,} municípios | população total: {df['POPULACAO'].sum():,.0f}")
    return df


for ano in ANOS:
    fetch_populacao_municipal(ano)

print(f"\nPopulação em: {IBGE_DIR}")

## 4. PIB Municipal — IBGE Sidra Tabela 5938

**Tabela 5938** — Produto Interno Bruto dos Municípios.  
**Variável 37** — PIB total a preços correntes (R$ mil).

PIB per capita não vem direto na tabela; será calculado no notebook 2 dividindo pelo dado de população (Seção 3).

In [ ]:
def fetch_pib_municipal(ano: int) -> pd.DataFrame:
    cache = IBGE_DIR / f"pib_municipal_{ano}.csv"
    if cache.exists():
        return pd.read_csv(cache, dtype={"CODMUN7": str, "CODMUN6": str})

    url = f"https://apisidra.ibge.gov.br/values/t/5938/n6/all/v/37/p/{ano}"
    print(f"  → Sidra: {url}")
    r = requests.get(url, timeout=300)
    r.raise_for_status()
    raw = r.json()
    if not raw or len(raw) < 2:
        raise RuntimeError(f"Resposta vazia do Sidra para PIB/{ano}")

    df = pd.DataFrame(raw[1:])
    df = df.rename(columns={"D1C": "CODMUN7", "D1N": "MUNICIPIO_NOME", "V": "PIB_RS_MIL"})
    df["PIB_RS_MIL"] = pd.to_numeric(df["PIB_RS_MIL"], errors="coerce")
    df["ANO"]        = ano
    df["CODMUN7"]    = df["CODMUN7"].astype(str).str.strip()
    df["CODMUN6"]    = df["CODMUN7"].str[:6]
    df = df[["CODMUN7", "CODMUN6", "MUNICIPIO_NOME", "ANO", "PIB_RS_MIL"]]
    df.to_csv(cache, index=False)
    print(f"     {len(df):,} municípios | PIB total: R$ {df['PIB_RS_MIL'].sum()/1_000_000:,.1f} bi")
    return df


# Tab 5938 só vai até 2023 (lag de 2 anos é normal). Se ANO solicitado não existe, cai para o mais recente.
ANOS_PIB_DISPONIVEIS = list(range(2002, 2024))
for ano in ANOS:
    ano_efetivo = ano if ano in ANOS_PIB_DISPONIVEIS else max(a for a in ANOS_PIB_DISPONIVEIS if a <= ano)
    if ano_efetivo != ano:
        print(f"  PIB de {ano} indisponível, usando {ano_efetivo}")
    fetch_pib_municipal(ano_efetivo)

print(f"\nPIB em: {IBGE_DIR}")

## 5. Despesa em Saúde — SICONFI / Tesouro Nacional

**API DCA** (Demonstrações Contábeis Anuais), **Anexo I-E** (Despesas por Função).  
Função CID-10 das contas: **`10 - Saúde`** → linhas com despesas empenhadas, liquidadas, pagas e restos a pagar.

**Endpoint:** `https://apidatalake.tesouro.gov.br/ords/siconfi/tt/dca`

A API exige uma chamada por município/ano. Para acelerar, usamos `ThreadPoolExecutor` com 10 workers (~5 minutos para todo o Brasil em um ano).

In [ ]:
SICONFI_URL = "https://apidatalake.tesouro.gov.br/ords/siconfi/tt/dca"


def listar_municipios_ibge() -> list[dict]:
    """Lista todos os municípios com seus códigos IBGE 7 dígitos via SICONFI."""
    cache = SICONFI_DIR / "municipios.json"
    if cache.exists():
        return json.loads(cache.read_text(encoding="utf-8"))
    r = requests.get("https://apidatalake.tesouro.gov.br/ords/siconfi/tt/entes", timeout=60)
    r.raise_for_status()
    municipios = [e for e in r.json()["items"] if e.get("esfera") == "M"]
    cache.write_text(json.dumps(municipios, ensure_ascii=False), encoding="utf-8")
    return municipios


def fetch_despesa_saude_municipio(cod_ibge: int, ano: int) -> list[dict]:
    """Busca todas as linhas de Despesa por Função = '10 - Saúde' para um município/ano."""
    cache = SICONFI_DIR / f"_mun_{cod_ibge}_{ano}.json"
    if cache.exists():
        return json.loads(cache.read_text(encoding="utf-8"))

    params = {"an_exercicio": ano, "no_anexo": "DCA-Anexo I-E", "id_ente": cod_ibge}
    try:
        r = requests.get(SICONFI_URL, params=params, timeout=60)
        r.raise_for_status()
        items = r.json().get("items", [])
    except Exception:
        items = []

    saude = [
        {
            "COD_IBGE":   x.get("cod_ibge"),
            "UF":         x.get("uf"),
            "ANO":        x.get("exercicio"),
            "COLUNA":     x.get("coluna"),
            "VALOR":      x.get("valor"),
            "POPULACAO":  x.get("populacao"),
        }
        for x in items if (x.get("conta") or "").strip() == "10 - Saúde"
    ]
    cache.write_text(json.dumps(saude, ensure_ascii=False), encoding="utf-8")
    return saude


def fetch_despesa_saude_ano(ano: int, max_workers: int = 10) -> pd.DataFrame:
    """Busca despesa em saúde de todos os municípios para um ano. Concorrente."""
    csv_path = SICONFI_DIR / f"despesa_saude_{ano}.csv"
    if csv_path.exists():
        print(f"  [cache] {csv_path.name}")
        return pd.read_csv(csv_path)

    municipios = listar_municipios_ibge()
    print(f"  Buscando despesa em saúde de {len(municipios)} municípios para {ano}...")
    t0 = time.time()
    todos = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(fetch_despesa_saude_municipio, m["cod_ibge"], ano): m for m in municipios}
        feitos = 0
        for fut in as_completed(futures):
            todos.extend(fut.result())
            feitos += 1
            if feitos % 500 == 0:
                print(f"     {feitos}/{len(municipios)} ({(time.time()-t0):.0f}s)")

    df = pd.DataFrame(todos)
    df.to_csv(csv_path, index=False)
    # Limpa caches por município (já consolidados no CSV)
    for f in SICONFI_DIR.glob(f"_mun_*_{ano}.json"):
        f.unlink()
    print(f"  Concluído em {(time.time()-t0):.0f}s — {len(df):,} linhas → {csv_path.name}")
    return df


for ano in ANOS:
    fetch_despesa_saude_ano(ano)

print(f"\nSICONFI em: {SICONFI_DIR}")

## 6. Atlas DH — IDH-M, Gini, Renda por quinto via API IPEA Data

**API:** `http://www.ipeadata.gov.br/api/odata4/`

Os dados do Atlas do Desenvolvimento Humano (PNUD/IPEA/FJP) — IDH-M, Gini, renda média por quinto, pobreza, analfabetismo, esperança de vida — estão **integralmente disponíveis na API do IPEA Data** sob códigos `ADH_*`. Usamos isso em vez do download manual em <atlasbrasil.org.br>.

**Ano de referência:** 2010 (último Censo com Atlas calculado; próxima rodada com Censo 2022 ainda não publicada).

**Tempo:** ~7 segundos (17 indicadores × 5.564 municípios em paralelo).

| Código IPEA | Coluna saída | Descrição |
|---|---|---|
| `ADH_IDHM` | `IDHM` | Índice de Desenvolvimento Humano Municipal |
| `ADH_IDHM_E` / `_L` / `_R` | `IDHM_E` / `IDHM_L` / `IDHM_R` | Sub-índices Educação / Longevidade / Renda |
| `ADH_GINI` | `GINI` | Coeficiente de Gini |
| `ADH_RDPC` | `RDPC` | Renda domiciliar per capita média |
| `ADH_RDPC1`–`RDPC4` | `RDPC1`–`RDPC4` | **Renda média do 1º ao 4º quinto mais pobre** |
| `ADH_RDPC5` | `RDPC5` | Renda média do quinto mais rico |
| `ADH_PMPOB` / `ADH_PIND` | `PMPOB` / `PIND` | % de pobres / extremamente pobres |
| `ADH_THEIL` | `THEIL` | Índice de Theil-L (desigualdade) |
| `ADH_ESPVIDA` | `ESPVIDA` | Esperança de vida ao nascer |
| `ADH_T_ANALF15M` | `T_ANALF15M` | Taxa de analfabetismo (15+ anos) |
| `ADH_T_AGUA` | `T_AGUA` | % com água encanada |

In [9]:
INDICADORES_IPEA = {
    "ADH_IDHM":       "IDHM",
    "ADH_IDHM_E":     "IDHM_E",
    "ADH_IDHM_L":     "IDHM_L",
    "ADH_IDHM_R":     "IDHM_R",
    "ADH_GINI":       "GINI",
    "ADH_RDPC":       "RDPC",
    "ADH_RDPC1":      "RDPC1",   # renda média do 1º quinto mais pobre
    "ADH_RDPC2":      "RDPC2",
    "ADH_RDPC3":      "RDPC3",
    "ADH_RDPC4":      "RDPC4",
    "ADH_RDPC5":      "RDPC5",   # renda média do quinto mais rico
    "ADH_PMPOB":      "PMPOB",
    "ADH_PIND":       "PIND",
    "ADH_THEIL":      "THEIL",
    "ADH_ESPVIDA":    "ESPVIDA",
    "ADH_T_ANALF15M": "T_ANALF15M",
    "ADH_T_AGUA":     "T_AGUA",
}
ANO_ATLAS = 2010


def fetch_ipea_municipios(serie_codigo: str, ano: int = 2010) -> list[dict]:
    """Busca uma série IPEA Data filtrada por município/ano."""
    url = f"http://www.ipeadata.gov.br/api/odata4/ValoresSerie(SERCODIGO='{serie_codigo}')"
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    items = r.json()["value"]
    return [d for d in items
            if "unic" in d.get("NIVNOME", "")
            and (d.get("VALDATA") or "")[:4] == str(ano)]


def fetch_atlas_dh_ipea(ano: int = 2010) -> pd.DataFrame:
    """Baixa todos os indicadores do Atlas via IPEA em paralelo. Cache local."""
    ATLAS_FILE = ATLAS_DIR / f"atlas_dh_ipea_{ano}.csv"
    if ATLAS_FILE.exists():
        return pd.read_csv(ATLAS_FILE, dtype={"CODMUN7": str, "CODMUN6": str})

    print(f"  Baixando {len(INDICADORES_IPEA)} indicadores via IPEA Data (paralelo)...")
    t0 = time.time()
    resultados = {}
    with ThreadPoolExecutor(max_workers=8) as ex:
        futures = {ex.submit(fetch_ipea_municipios, cod, ano): cod
                   for cod in INDICADORES_IPEA}
        for fut in as_completed(futures):
            cod = futures[fut]
            resultados[cod] = fut.result()
            print(f"    {cod:<22} {INDICADORES_IPEA[cod]:<12} {len(resultados[cod])} municípios")

    # Pivota: 1 linha por município, 1 coluna por indicador
    registros: dict[str, dict] = {}
    for cod, items in resultados.items():
        nome = INDICADORES_IPEA[cod]
        for d in items:
            cod7 = d["TERCODIGO"]
            if cod7 not in registros:
                registros[cod7] = {"CODMUN7": cod7}
            registros[cod7][nome] = d["VALVALOR"]

    df = pd.DataFrame(list(registros.values()))
    df["CODMUN6"] = df["CODMUN7"].str[:6]
    cols = ["CODMUN7", "CODMUN6"] + list(INDICADORES_IPEA.values())
    df = df[[c for c in cols if c in df.columns]]
    df.to_csv(ATLAS_FILE, index=False)
    print(f"  Concluído em {time.time()-t0:.1f}s — {len(df):,} municípios → {ATLAS_FILE.name}")
    return df


df_atlas = fetch_atlas_dh_ipea(ANO_ATLAS)
print(f"Dimensões: {df_atlas.shape}")
df_atlas.head(3)

  Baixando 17 indicadores via IPEA Data (paralelo)...
    ADH_RDPC               RDPC         5564 municípios
    ADH_GINI               GINI         5564 municípios
    ADH_RDPC2              RDPC2        5564 municípios
    ADH_RDPC1              RDPC1        5564 municípios
    ADH_IDHM               IDHM         5564 municípios
    ADH_IDHM_E             IDHM_E       5564 municípios
    ADH_IDHM_L             IDHM_L       5564 municípios
    ADH_IDHM_R             IDHM_R       5564 municípios
    ADH_PIND               PIND         5564 municípios
    ADH_RDPC4              RDPC4        5564 municípios
    ADH_RDPC3              RDPC3        5564 municípios
    ADH_RDPC5              RDPC5        5564 municípios
    ADH_PMPOB              PMPOB        5564 municípios
    ADH_ESPVIDA            ESPVIDA      5564 municípios
    ADH_THEIL              THEIL        5564 municípios
    ADH_T_ANALF15M         T_ANALF15M   5564 municípios
    ADH_T_AGUA             T_AGUA       5564 munic

,CODMUN7,CODMUN6,IDHM,IDHM_E,IDHM_L,IDHM_R,GINI,RDPC,RDPC1,RDPC2,RDPC3,RDPC4,RDPC5,PMPOB,PIND,THEIL,ESPVIDA,T_ANALF15M,T_AGUA
0,1200203,120020,0.664,0.582,0.776,0.648,0.64,450.06,28.67,120.84,222.94,387.63,1493.27,34.51,18.08,0.74,71.58,18.52,79.14
1,1200336,120033,0.625,0.546,0.770,0.580,0.61,295.50,15.37,78.28,157.50,288.28,938.10,45.68,28.99,0.69,71.17,23.79,80.26
2,1200351,120035,0.501,0.361,0.726,0.479,0.59,157.27,3.33,42.64,97.49,168.20,474.49,63.82,39.31,0.51,68.55,34.38,50.76


## 7. Resumo dos arquivos extraídos

In [10]:
def listar(diretorio: Path, padrao: str = "*") -> list[tuple[str, str]]:
    arquivos = sorted(diretorio.glob(padrao))
    return [(f.name, f"{f.stat().st_size / 1024:.1f} KB") for f in arquivos if f.is_file()]


print(f"=== SIM ({SIM_DIR}) ===")
for nome, tam in listar(SIM_DIR, "*.parquet"):
    print(f"  {nome:<25} {tam}")

print(f"\n=== IBGE ({IBGE_DIR}) ===")
for nome, tam in listar(IBGE_DIR, "*.csv"):
    print(f"  {nome:<35} {tam}")

print(f"\n=== SICONFI ({SICONFI_DIR}) ===")
for nome, tam in listar(SICONFI_DIR, "*.csv"):
    print(f"  {nome:<35} {tam}")

print(f"\n=== Atlas ({ATLAS_DIR}) ===")
for nome, tam in listar(ATLAS_DIR, "*"):
    print(f"  {nome:<35} {tam}")

=== SIM (c:\Users\cerb3\OneDrive\Área de Trabalho\IESB\Tópicos BD\data\raw\sim) ===
  DOAC2022.parquet          296.3 KB
  DOAL2022.parquet          1290.0 KB
  DOAM2022.parquet          1177.5 KB
  DOAP2022.parquet          280.2 KB
  DOBA2022.parquet          5412.2 KB
  DOCE2022.parquet          3397.7 KB
  DODF2022.parquet          917.3 KB
  DOES2022.parquet          1533.6 KB
  DOGO2022.parquet          2786.7 KB
  DOMA2022.parquet          2174.5 KB
  DOMG2022.parquet          8075.8 KB
  DOMS2022.parquet          1167.1 KB
  DOMT2022.parquet          1348.9 KB
  DOPA2022.parquet          2547.1 KB
  DOPB2022.parquet          1705.3 KB
  DOPE2022.parquet          3685.4 KB
  DOPI2022.parquet          1358.0 KB
  DOPR2022.parquet          4880.4 KB
  DORJ2022.parquet          7092.0 KB
  DORN2022.parquet          1340.2 KB
  DORO2022.parquet          656.1 KB
  DORR2022.parquet          258.6 KB
  DORS2022.parquet          5111.1 KB
  DOSC2022.parquet          2790.7 KB
  DOSE202

---

**Próximo passo:** abra `02_transformacao.ipynb` para aplicar a Lista Brasileira de Causas Evitáveis (LBCE), agregar óbitos por município e construir a matriz de features para a regressão.